In [1]:
from pathlib import Path

DATA_DIR = Path("../data")
DICTIONARY_DIR = Path("../dictionaries")

if not DATA_DIR.exists():
    DATA_DIR = Path("data")

if not DICTIONARY_DIR.exists():
    DICTIONARY_DIR = Path("dictionaries")

SAMOKAT_MAIN_PATH = DATA_DIR / "samokat_esci.csv"

DOMAIN_DICTIONARY_PATH = DICTIONARY_DIR / "domain_dictionary.txt"


In [2]:
import pandas as pd

df = pd.read_csv(SAMOKAT_MAIN_PATH, encoding='utf-8')

preprocessing = df.copy()
preprocessing['query'] = preprocessing['query'].str.lower().str.replace(
    r'[^\p{L}\s-]', '', regex=True
)

preprocessing['item_name'] = (
    preprocessing['item_name']
    .str.split(',').str[0]
    .str.replace(r'[^а-яА-Яa-zA-Z\s-]', '', regex=True)
    .str.replace(r'\b(шт|г|кг|мл|л)\b', '', regex=True)
)

In [3]:
non_eat = set(['уход', 'детская гигиена и уход', 'одежда', 'продукция для животных',
                'гигиена', 'спорт и активный отдых', 'книги, хобби, канцелярия', 
                'обувь', 'детские товары', 'смартфоны и гаджеты', 'товары для дома',
                'аптека', 'ремонт и обустройство дома', 'бытовая химия', 'парфюмерия и декоративная косметика',
                'компьютеры и аксессуары', 'аксессуары', 'праздничные и сезонные товары', 'посуда и кухонные принадлежности',
                'бытовая техника и электроника', 'автомобильные товары и запчасти', 'мебель',
                'строительство и ремонт', 'хозяйственные товары', 'комплектующие для продажи',
                'товары для взрослых', 'образцы товаров', 'товары для пикника', 'маркетинг товары промо',
                'имущество магазина', 'бады и витамины', 'табачные изделия'
])

eat = set(['безалкогольные напитки', 'кулинария', 'бакалея', 
            'молочная продукция', 'снэки', 'птица', 'мороженое и десерты',
            'мучные кондитерские изделия', 'кофе, какао', 'мясная гастрономия',
            'хлеб и хлебобулочные изделия', 'замороженные продукты', 'кондитерские изделия',
            'сыры', 'свежие овощи', 'детское питание', 'консервированные продукты',
            'рыбная гастрономия', 'чай', 'специальное питание',
            'слабоалкогольные напитки', 'мясо', 'свежие фрукты', 'яичные товары',
            'рынок', 'алкоголь',  'рыба охлажденная', 

])

предобработка: оставляем только строки с e/s уникальным классом

In [4]:
def category(q):
    non_eat = set(['уход', 'детская гигиена и уход', 'одежда', 'продукция для животных',
                'гигиена', 'спорт и активный отдых', 'книги, хобби, канцелярия', 
                'обувь', 'детские товары', 'смартфоны и гаджеты', 'товары для дома',
                'аптека', 'ремонт и обустройство дома', 'бытовая химия', 'парфюмерия и декоративная косметика',
                'компьютеры и аксессуары', 'аксессуары', 'праздничные и сезонные товары', 'посуда и кухонные принадлежности',
                'бытовая техника и электроника', 'автомобильные товары и запчасти', 'мебель',
                'строительство и ремонт', 'хозяйственные товары', 'комплектующие для продажи',
                'товары для взрослых', 'образцы товаров', 'товары для пикника', 'маркетинг товары промо',
                'имущество магазина', 'бады и витамины', 'табачные изделия'
])
    if q in non_eat: 
        return 'не еда'
    return 'еда'

In [5]:
item_names = preprocessing[['item_name', 'category1_name']]

item_names = preprocessing[['item_name', 'category1_name']].rename(columns={'item_name': 'query'})
item_names = item_names.dropna(subset=['category1_name']).drop_duplicates(subset=['query'])
item_names['category0_name'] = item_names['category1_name'].apply(category)


In [6]:
rows = preprocessing[preprocessing['final_answer'].isin(['e', 's'])]
rows = rows.dropna(subset='category1_name')

category_counts = rows.groupby('query')['category1_name'].nunique()
clean_queries = category_counts[category_counts == 1].index

category1_dataset = (
    rows[rows['query'].isin(clean_queries)]
    .drop_duplicates(subset=['query'])
    [['query', 'category1_name']]
)


category1_dataset['category0_name'] = category1_dataset['category1_name'].apply(lambda q: category(q))

# category1_dataset = pd.concat([category1_dataset, item_names], ignore_index=True)

In [66]:
def build_dataset(df, key_col, unique_category_only=False):
    d = df.dropna(subset=['category1_name'])
    if unique_category_only:
        clean_keys = d.groupby(key_col)['category1_name'].nunique().loc[lambda s: s == 1].index
        d = d[d[key_col].isin(clean_keys)]
    d = d.drop_duplicates(subset=[key_col])[[key_col, 'category1_name']].rename(columns={key_col: 'query'})
    d['category0_name'] = d['category1_name'].apply(category)
    return d

item_names = build_dataset(preprocessing, 'item_name')

rows = preprocessing[preprocessing['final_answer'].isin(['e', 's'])]
category1_dataset = build_dataset(rows, 'query', unique_category_only=True)

category1_dataset = pd.concat([category1_dataset, item_names], ignore_index=True)


In [7]:
class_counts = category1_dataset['category1_name'].value_counts()
valid_classes = class_counts[class_counts >= 3].index
category1_dataset = category1_dataset[category1_dataset['category1_name'].isin(valid_classes)]


In [8]:
from sklearn.model_selection import train_test_split

Один классификатор

In [9]:
X_train = item_names['query']
y_train = item_names['category1_name']

X_test = category1_dataset['query']
y_test = category1_dataset['category1_name']

# валидация - выделяем из train
X_train, X_val, y_train, y_val = train_test_split(
    X_train,
    y_train,
    stratify=y_train,
    test_size=0.176,
    random_state=42
)



эмбеддинги

In [10]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer('d0rj/e5-small-en-ru')

X_train = [f"query: {q}" for q in X_train]
X_test = [f"query: {q}" for q in X_test]
X_val = [f"query: {q}" for q in X_val]

X_train_emb = embedder.encode(X_train, convert_to_numpy=True)
X_test_emb = embedder.encode(X_test, convert_to_numpy=True)
X_val_emb = embedder.encode(X_val, convert_to_numpy=True)

/opt/anaconda3/envs/study/lib/python3.11/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


In [11]:
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier

kn_model = KNeighborsClassifier(weights='distance', n_neighbors=3, metric='euclidean')
log_model = LogisticRegression(class_weight='balanced', max_iter=1000)

kn_model.fit(X_train_emb, y_train)
log_model.fit(X_train_emb, y_train)

y_val_pred_kn = kn_model.predict(X_val_emb)
y_val_pred_log = log_model.predict(X_val_emb)


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [12]:
from sklearn.model_selection import GridSearchCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsClassifier())
])

param_grid = {
    'knn__n_neighbors': [3, 5, 7, 9, 11, 15],
    'knn__weights': ['uniform', 'distance'],
    'knn__metric': ['euclidean', 'manhattan']
}

grid_search = GridSearchCV(
    estimator=pipeline, 
    param_grid=param_grid, 
    cv=5, 
    scoring='accuracy',
    n_jobs=-1           
)

grid_search.fit(X_train_emb, y_train)

print("лучшие параметры:", grid_search.best_params_)
print("лучшая точность:", grid_search.best_score_)


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
/opt/anaconda3/envs/study/lib/python3.11/site-packages/sklearn/model_selection/_split.py:813: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=5.
  warnings.warn(
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...


лучшие параметры: {'knn__metric': 'euclidean', 'knn__n_neighbors': 3, 'knn__weights': 'distance'}
лучшая точность: 0.8657240370550949


In [13]:
from sklearn.metrics import accuracy_score, f1_score

accuracy_log = accuracy_score(y_val, y_val_pred_log)
f1_macro_log = f1_score(y_val, y_val_pred_log, average="macro")

accuracy_kn = accuracy_score(y_val, y_val_pred_kn)
f1_macro_kn = f1_score(y_val, y_val_pred_kn, average='macro')

print(f"log accuracy: {accuracy_log}\nlog f1_macro: {f1_macro_log}")
print(f"kn accuracy: {accuracy_kn}\nkn f1_macro: {f1_macro_kn}")

log accuracy: 0.6836338735448527
log f1_macro: 0.5769354315049313
kn accuracy: 0.8806208628167085
kn f1_macro: 0.7428632105932012


In [92]:
import plotly.graph_objects as go
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_val, y_val_pred_kn, labels=kn_model.classes_)
cm_normalized = cm / cm.sum(axis=1, keepdims=True)

brand_colorscale = [
    [0.00, "#FFFFFF"],   
    [0.01, "#FF8FA3"],   
    [0.35, "#FF3357"],
    [0.60, "#CC0024"],
    [0.80, "#99001B"],
    [1.00, "#660012"],   
]

fig = go.Figure(data=go.Heatmap(
    z=cm_normalized,
    x=kn_model.classes_,
    y=kn_model.classes_,
    colorscale=brand_colorscale,
    zmin=0, zmax=1,
    hovertemplate='истинный класс: %{y}<br>Предсказано: %{x}<br>Доля: %{z:.1%}<extra></extra>',
    colorbar=dict(title='Доля (recall)')
))

fig.update_layout(
    title='матрица ошибок',
    xaxis_title='предсказанный класс',
    yaxis_title='истинный класс',
    xaxis=dict(tickangle=90),
    width=950, height=950,
    plot_bgcolor='white',
)
fig.update_yaxes(autorange='reversed')
fig.show()



/var/folders/qg/d74931cs1h7gmtnh53dz3tbh0000gn/T/ipykernel_28894/3318807948.py:5: RuntimeWarning: invalid value encountered in divide
  cm_normalized = cm / cm.sum(axis=1, keepdims=True)


построить категории выше и объедигнить туда - два классификатора


In [16]:
import pandas as pd
from sklearn.metrics import classification_report

report_dict = classification_report(y_val, y_val_pred_kn, output_dict=True)

df_report = pd.DataFrame(report_dict).T

system_rows = ["accuracy", "macro avg", "weighted avg"]
df_report = df_report.drop(index=system_rows, errors="ignore")

worst_classes = df_report.sort_values(by="f1-score", ascending=True).head(10)

print("--- худшие классы ---")
print(worst_classes[["precision", "recall", "f1-score", "support"]].round(3))


--- худшие классы ---
                                  precision  recall  f1-score  support
образцы товаров                       0.000   0.000     0.000      1.0
рынок                                 0.000   0.000     0.000      3.0
имущество магазина                    0.000   0.000     0.000      1.0
комплектующие для продажи             0.000   0.000     0.000      2.0
рыба охлажденная                      0.000   0.000     0.000      1.0
товары для пикника                    0.000   0.000     0.000      1.0
слабоалкогольные напитки              1.000   0.333     0.500      3.0
посуда и кухонные принадлежности      0.417   0.714     0.526      7.0
товары для взрослых                   0.714   0.500     0.588     10.0
строительство и ремонт                0.714   0.500     0.588     10.0


/opt/anaconda3/envs/study/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/anaconda3/envs/study/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/anaconda3/envs/study/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.sh

теперь попробуем обучить два классификатора

In [46]:
X = category1_dataset['query']
y = category1_dataset['category0_name']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.15,
    random_state=42,
    stratify=y
)

# валидация

X_train, X_val, y_train, y_val = train_test_split(
    X_train,
    y_train, 
    stratify=y_train,
    test_size=0.176,
    random_state=42
)

In [ ]:
embedder = SentenceTransformer('d0rj/e5-small-en-ru')

X_train = [f"query: {q}" for q in X_train]
X_test = [f"query: {q}" for q in X_test]
X_val = [f"query: {q}" for q in X_val]

X_train_emb = embedder.encode(X_train, convert_to_numpy=True)
X_test_emb = embedder.encode(X_test, convert_to_numpy=True)
X_val_emb = embedder.encode(X_val, convert_to_numpy=True)

'(MaxRetryError('HTTPSConnectionPool(host=\'huggingface.co\', port=443): Max retries exceeded with url: /d0rj/e5-small-en-ru/resolve/main/modules.json (Caused by NameResolutionError("HTTPSConnection(host=\'huggingface.co\', port=443): Failed to resolve \'huggingface.co\' ([Errno 8] nodename nor servname provided, or not known)"))'), '(Request ID: 2b86a153-c7e1-4a0f-a3ec-d74ec667bbc2)')' thrown while requesting HEAD https://huggingface.co/d0rj/e5-small-en-ru/resolve/main/./modules.json
Retrying in 1s [Retry 1/5].


'(MaxRetryError('HTTPSConnectionPool(host=\'huggingface.co\', port=443): Max retries exceeded with url: /d0rj/e5-small-en-ru/resolve/main/modules.json (Caused by NameResolutionError("HTTPSConnection(host=\'huggingface.co\', port=443): Failed to resolve \'huggingface.co\' ([Errno 8] nodename nor servname provided, or not known)"))'), '(Request ID: ed40a551-b1b5-4d9d-984e-900d53794427)')' thrown while requesting HEAD https://huggingface.co/d0rj/e5-small-en-ru/resolve/main/./modules.json
Retrying in 2s [Retry 2/5].
'(MaxRetryError('HTTPSConnectionPool(host=\'huggingface.co\', port=443): Max retries exceeded with url: /d0rj/e5-small-en-ru/resolve/main/modules.json (Caused by NameResolutionError("HTTPSConnection(host=\'huggingface.co\', port=443): Failed to resolve \'huggingface.co\' ([Errno 8] nodename nor servname provided, or not known)"))'), '(Request ID: 01654e93-704c-4e7d-8a79-817dbe72e976)')' thrown while requesting HEAD https://huggingface.co/d0rj/e5-small-en-ru/resolve/main/./modul

Дообучение

In [ ]:
from datasets import Dataset
from sklearn.preprocessing import LabelEncoder
from sentence_transformers.losses import BatchHardTripletLoss
from sentence_transformers import SentenceTransformerTrainingArguments
from sentence_transformers.training_args import BatchSamplers

encoder = LabelEncoder()
loss = BatchHardTripletLoss(model=embedder, margin=4)

y_train_encoded = encoder.fit_transform(y_train)

train_dataset = Dataset.from_dict({
    'sentence': X_train,
    'label': y_train_encoded 
})


args = SentenceTransformerTrainingArguments(
    output_dir="./finetuned-e5",
    num_train_epochs=3,
    per_device_train_batch_size=32,
    batch_sampler=BatchSamplers.GROUP_BY_LABEL,
)

In [ ]:
from sentence_transformers import SentenceTransformerTrainer

trainer = SentenceTransformerTrainer(
    model=embedder,
    args=args,
    train_dataset=train_dataset,
    loss=loss,
)

trainer.train()


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 1}.
/opt/anaconda3/envs/study/lib/python3.11/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=339, training_loss=4.000471357047382, metrics={'train_runtime': 178.8129, 'train_samples_per_second': 61.288, 'train_steps_per_second': 1.896, 'total_flos': 0.0, 'train_loss': 4.000471357047382, 'epoch': 3.0})

In [ ]:
X_train_emb = embedder.encode(X_train, convert_to_numpy=True)
X_test_emb = embedder.encode(X_test, convert_to_numpy=True)
X_val_emb = embedder.encode(X_val, convert_to_numpy=True)

accuracy = accuracy_score(y_val, y_val_pred)
f1_macro = f1_score(y_val, y_val_pred, average="macro")

print(f"accuracy: {accuracy}\nf1_macro: {f1_macro}")

accuracy: 0.70550576184379
f1_macro: 0.513294717458871


дообучение неэффективно

попробовать гигачат для эмбеддингов
попробовать построить несколько классификаторов с категориями повыше

можно попробовать обучить на item_name - там все e 